In [ ]:
# Configure the notebook environment

import syspre
import boto3
import pandas as pd
import numpy as np
import sklearn
import sagemaker

print("Python:", sys.version)
print("Boto3:", boto3.__version__)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("SageMaker SDK:", sagemaker.__version__)

Python: 3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 16:50:00) [GCC 14.3.0]
Boto3: 1.43.0
Pandas: 3.0.5
NumPy: 2.4.6
Scikit-learn: 1.5.2
SageMaker SDK: 2.257.5


In [ ]:
# Configure AWS and S3

import boto3
import sagemaker
from sagemaker import get_execution_role

SAGEMAKER_SUPPRESS_V2_WARNING=1

region = boto3.Session().region_name
role = get_execution_role()
sm_session = sagemaker.Session()

bucket = "sagemaker-us-east-1-581187100103"
project_prefix = "umass/customer-churn"

raw_s3_uri = f"s3://{bucket}/{project_prefix}/data/raw/storedata_total.csv"
processed_prefix = f"s3://{bucket}/{project_prefix}/data/processed"
model_output_path = f"s3://{bucket}/{project_prefix}/model-artifacts"
evaluation_output_path = f"s3://{bucket}/{project_prefix}/evaluation"
batch_output_path = f"s3://{bucket}/{project_prefix}/batch-output"

print("Region:", region)
print("Role:", role)
print("Bucket:", bucket)
print("Raw data:", raw_s3_uri)

Region: us-east-1
Role: arn:aws:iam::581187100103:role/service-role/AmazonSageMaker-ExecutionRole-20260723T215584
Bucket: sagemaker-us-east-1-581187100103
Raw data: s3://sagemaker-us-east-1-581187100103/umass/customer-churn/data/raw/storedata_total.csv


In [ ]:
# Test access

s3 = boto3.client("s3")
s3.head_bucket(Bucket=bucket)
print("S3 bucket is accessible.")

S3 bucket is accessible.


In [ ]:
# Collect the data

local_data_path = "../data/storedata_total.csv"

sm_session.upload_data(
    path=local_data_path,
    bucket=bucket,
    key_prefix=f"{project_prefix}/data/raw"
)

print(raw_s3_uri)

s3://sagemaker-us-east-1-581187100103/umass/customer-churn/data/raw/storedata_total.csv


In [35]:
# Verify from the notebook

response = s3.head_object(
    Bucket=bucket,
    Key=f"{project_prefix}/data/raw/storedata_total.csv"
)

print("File size:", response["ContentLength"], "bytes")

File size: 3394535 bytes


In [36]:
# Load and inspect the raw data

import pandas as pd

df = pd.read_csv(raw_s3_uri)

print("Shape:", df.shape)
display(df.head())
df.info()

Shape: (30801, 15)


,custid,retained,created,firstorder,lastorder,esent,eopenrate,eclickrate,avgorder,ordfreq,paperless,refill,doorstep,favday,city
0,6H6T6N,0,2012-09-28,2013-08-11 00:00:00,2013-08-11 00:00:00,29,100.000000,3.448276,14.52,0.000000,0,0,0,Monday,DEL
1,APCENR,1,2010-12-19,2011-04-01 00:00:00,2014-01-19 00:00:00,95,92.631579,10.526316,83.69,0.181641,1,1,1,Friday,DEL
2,7UP6MS,0,2010-10-03,2010-12-01 00:00:00,2011-07-06 00:00:00,0,0.000000,0.000000,33.58,0.059908,0,0,0,Wednesday,DEL
3,7ZEW8G,0,2010-10-22,2011-03-28 00:00:00,2011-03-28 00:00:00,0,0.000000,0.000000,54.96,0.000000,0,0,0,Thursday,BOM
4,8V726M,1,2010-11-27,2010-11-29 00:00:00,2013-01-28 00:00:00,30,90.000000,13.333333,111.91,0.008850,0,0,0,Monday,BOM


<class 'pandas.DataFrame'>
RangeIndex: 30801 entries, 0 to 30800
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   custid      30781 non-null  str    
 1   retained    30801 non-null  int64  
 2   created     30781 non-null  str    
 3   firstorder  30781 non-null  str    
 4   lastorder   30781 non-null  str    
 5   esent       30801 non-null  int64  
 6   eopenrate   30801 non-null  float64
 7   eclickrate  30801 non-null  float64
 8   avgorder    30801 non-null  float64
 9   ordfreq     30801 non-null  float64
 10  paperless   30801 non-null  int64  
 11  refill      30801 non-null  int64  
 12  doorstep    30801 non-null  int64  
 13  favday      30801 non-null  str    
 14  city        30801 non-null  str    
dtypes: float64(4), int64(5), str(6)
memory usage: 5.4 MB


In [27]:
# Inspect columns

print(df.columns.tolist())

['custid', 'retained', 'created', 'firstorder', 'lastorder', 'esent', 'eopenrate', 'eclickrate', 'avgorder', 'ordfreq', 'paperless', 'refill', 'doorstep', 'favday', 'city']


In [28]:
# Inspect target

print(df["retained"].value_counts())
print(df["retained"].value_counts(normalize=True))

retained
1    24472
0     6329
Name: count, dtype: int64
retained
1    0.79452
0    0.20548
Name: proportion, dtype: float64


In [37]:
# Check data quality

quality_summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique()
})

display(quality_summary)

,dtype,missing_count,missing_percent,unique_values
custid,str,20,0.06,30769
retained,int64,0,0.00,2
created,str,20,0.06,2821
firstorder,str,20,0.06,2672
lastorder,str,20,0.06,2414
esent,int64,0,0.00,90
eopenrate,float64,0,0.00,977
eclickrate,float64,0,0.00,503
avgorder,float64,0,0.00,9937
ordfreq,float64,0,0.00,4388


In [38]:
# Check duplicates

print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [39]:
# Test preprocessing locally first

import sys
sys.path.append("../src")

from preprocess import preprocess_data

raw_df = pd.read_csv(raw_s3_uri)
prepared_df = preprocess_data(raw_df)

print("Raw shape:", raw_df.shape)
print("Prepared shape:", prepared_df.shape)
display(prepared_df.head())

Raw shape: (30801, 15)
Prepared shape: (30747, 22)


,retained,esent,eopenrate,eclickrate,avgorder,ordfreq,paperless,refill,doorstep,first_last_days_diff,...,favday_Monday,favday_Saturday,favday_Sunday,favday_Thursday,favday_Tuesday,favday_Wednesday,city_BLR,city_BOM,city_DEL,city_MAA
0,0,29,100.000000,3.448276,14.52,0.000000,0,0,0,0,...,1,0,0,0,0,0,0,0,1,0
1,1,95,92.631579,10.526316,83.69,0.181641,1,1,1,1024,...,0,0,0,0,0,0,0,0,1,0
2,0,0,0.000000,0.000000,33.58,0.059908,0,0,0,217,...,0,0,0,0,0,1,0,0,1,0
3,0,0,0.000000,0.000000,54.96,0.000000,0,0,0,0,...,0,0,0,1,0,0,0,1,0,0
4,1,30,90.000000,13.333333,111.91,0.008850,0,0,0,791,...,1,0,0,0,0,0,0,1,0,0


In [40]:
# Validate the output

assert "retained" in prepared_df.columns
assert prepared_df["retained"].isin([0, 1]).all()
assert prepared_df.isna().sum().sum() == 0

print("Preprocessing validation passed.")

Preprocessing validation passed.


In [41]:
# Review the engineered features

display(
    prepared_df[
        [
            "retained",
            "first_last_days_diff",
            "created_first_days_diff"
        ]
    ].describe()
)

,retained,first_last_days_diff,created_first_days_diff
count,30747.000000,30747.000000,30747.000000
mean,0.794647,90.582528,41.094969
std,0.403966,223.183471,125.897752
min,0.000000,0.000000,-39.000000
25%,1.000000,0.000000,0.000000
50%,1.000000,0.000000,1.000000
75%,1.000000,46.000000,22.000000
max,1.000000,1985.000000,1998.000000


In [43]:
# Run managed SageMaker Processing
#
# Create an SKLearn processor

SAGEMAKER_SUPPRESS_V2_WARNING = 1

from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

processing_instance_type = "ml.m5.large"

sklearn_processor = SKLearnProcessor(
    framework_version="1.4-2",
    role=role,
    instance_type=processing_instance_type,
    instance_count=1,
    base_job_name="umass-churn-preprocessing",
    sagemaker_session=sm_session
)


In [44]:
# Run preprocessing:

sklearn_processor.run(
    code="../src/preprocess.py",
    inputs=[
        ProcessingInput(
            source=raw_s3_uri,
            destination="/opt/ml/processing/input"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            source="/opt/ml/processing/train",
            destination=f"{processed_prefix}/train"
        ),
        ProcessingOutput(
            output_name="validation",
            source="/opt/ml/processing/validation",
            destination=f"{processed_prefix}/validation"
        ),
        ProcessingOutput(
            output_name="test",
            source="/opt/ml/processing/test",
            destination=f"{processed_prefix}/test"
        ),
        ProcessingOutput(
            output_name="metadata",
            source="/opt/ml/processing/metadata",
            destination=f"{processed_prefix}/metadata"
        )
    ],
    arguments=["--random-state", "42"],
    wait=True,
    logs=True
)

INFO:sagemaker:Creating processing-job with name umass-churn-preprocessing-2026-07-25-05-18-35-013


.

.

.

.

...

.

.

.


.

.

In [46]:
# Verify

description = sklearn_processor.latest_job.describe()

print("Job:", description["ProcessingJobName"])
print("Status:", description["ProcessingJobStatus"])
print("Failure reason:", description.get("FailureReason", "None"))

Job: umass-churn-preprocessing-2026-07-25-05-18-35-013
Status: Completed
Failure reason: None


In [47]:
for split in ["train", "validation", "test"]:
    key = f"{project_prefix}/data/processed/{split}/{split}.csv"
    result = s3.head_object(Bucket=bucket, Key=key)
    print(split, result["ContentLength"], "bytes")

train 1445436 bytes
validation 310150 bytes
test 310050 bytes


In [48]:
# Build an interactive baseline mode

from sagemaker import image_uris

xgb_image_uri = image_uris.retrieve(
    framework="xgboost",
    region=region,
    version="3.0-5",
    py_version="py3",
    instance_type="ml.m5.large"
)

print(xgb_image_uri)


INFO:sagemaker.image_uris:Ignoring unnecessary Python version: py3.


INFO:sagemaker.image_uris:Ignoring unnecessary instance type: ml.m5.large.


683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:3.0-5


In [49]:
# Create training inputs

from sagemaker.inputs import TrainingInput

train_input = TrainingInput(
    s3_data=f"{processed_prefix}/train",
    content_type="text/csv"
)

validation_input = TrainingInput(
    s3_data=f"{processed_prefix}/validation",
    content_type="text/csv"
)

In [52]:
# Create the estimator

from sagemaker.estimator import Estimator

xgb_estimator = Estimator(
    image_uri=xgb_image_uri,
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    volume_size=20,
    max_run=3600,
    output_path=model_output_path,
    base_job_name="umass-churn-xgboost",
    sagemaker_session=sm_session
)

xgb_estimator.set_hyperparameters(
    objective="binary:logistic",
    eval_metric="auc",
    num_round=200,
    eta=0.1,
    max_depth=5,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8
)

In [53]:
# Train

xgb_estimator.fit(
    {
        "train": train_input,
        "validation": validation_input
    },
    wait=True,
    logs=True
)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


INFO:sagemaker:Creating training-job with name: umass-churn-xgboost-2026-07-25-21-49-12-892


2026-07-25 21:49:14 Starting - Starting the training job.

.

.

...

.

.


2026-07-25 21:51:45 Training - Training image download completed. Training in progress.
2026-07-25 21:51:45 Uploading - Uploading generated training model/miniconda3/lib/python3.10/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-07-25:21:51:37:INFO] Imported framework sagemaker_xgboost_container.training
[2026-07-25:21:51:37:INFO] Failed to parse hyperparameter eval_metric value auc to Json.
Returning the value itself
[2026-07-25:21:51:37:INFO] Failed to parse hyperparameter objective value binary:logistic to Json.
Returning the value itself
[2026-07-25:21:51:37:INFO] No GPUs detected (normal if no gpus installed)
[2026-07-25:21:51:37:INFO] Running XGBoost Sagemaker in algorithm mode
[2026-07-25:21:51:37:INFO] Determ


2026-07-25 21:51:59 Completed - Training job completed


Training seconds: 125
Billable seconds: 125


In [54]:
# Verify

training_job = xgb_estimator.latest_training_job
description = training_job.describe()

print("Training job:", training_job.name)
print("Status:", description["TrainingJobStatus"])
print("Secondary status:", description["SecondaryStatus"])
print("Failure reason:", description.get("FailureReason", "None"))

Training job: umass-churn-xgboost-2026-07-25-21-49-12-892
Status: Completed
Secondary status: Completed
Failure reason: None


In [55]:
# Tune the model

from sagemaker.tuner import (
    HyperparameterTuner,
    ContinuousParameter,
    IntegerParameter
)

hyperparameter_ranges = {
    "eta": ContinuousParameter(0.01, 0.3, scaling_type="Logarithmic"),
    "max_depth": IntegerParameter(3, 10),
    "min_child_weight": ContinuousParameter(1, 10),
    "subsample": ContinuousParameter(0.6, 1.0),
    "colsample_bytree": ContinuousParameter(0.6, 1.0),
    "alpha": ContinuousParameter(0.0, 5.0),
    "lambda": ContinuousParameter(0.1, 10.0)
}


In [59]:
# Create the tuner

SAGEMAKER_SUPPRESS_V2_WARNING = 1

tuner = HyperparameterTuner(
    estimator=xgb_estimator,
    objective_metric_name="validation:auc",
    hyperparameter_ranges=hyperparameter_ranges,
    objective_type="Maximize",
    max_jobs=10,
    max_parallel_jobs=2,
    strategy="Bayesian",
    base_tuning_job_name="umass-churn-hpo"
)

In [60]:
# Run tuning

tuner.fit(
    {
        "train": train_input,
        "validation": validation_input
    },
    wait=True,
    logs=True
)

INFO:sagemaker:Creating hyperparameter tuning job with name: umass-churn-hpo-260725-2158


.

.

.

.

...

.

.

.

.

!

In [66]:
# Monitor progress

import boto3

sm_client = boto3.client("sagemaker", region_name=region)

# Find recent tuning jobs
jobs = sm_client.list_hyper_parameter_tuning_jobs(
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=10
)

for job in jobs["HyperParameterTuningJobSummaries"]:
    print(
        job["HyperParameterTuningJobName"],
        job["HyperParameterTuningJobStatus"]
    )

tuning_job_name = "umass-churn-hpo-260725-2158"

status = sm_client.describe_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=tuning_job_name
)

print("Status:", status["HyperParameterTuningJobStatus"])
print("Job counters:", status["TrainingJobStatusCounters"])
print("Failure reason:", status.get("FailureReason", "None"))


umass-churn-hpo-260725-2158 Completed
Status: Completed
Job counters: {'Completed': 10, 'InProgress': 0, 'RetryableError': 0, 'NonRetryableError': 0, 'Stopped': 0}
Failure reason: None


In [67]:
# Review results

analytics = tuner.analytics()
tuning_results = analytics.dataframe()

display(
    tuning_results.sort_values(
        "FinalObjectiveValue",
        ascending=False
    ).head(10)
)

,alpha,colsample_bytree,eta,lambda,max_depth,min_child_weight,subsample,TrainingJobName,TrainingJobStatus,FinalObjectiveValue,TrainingStartTime,TrainingEndTime,TrainingElapsedTimeSeconds
1,2.465210,0.794429,0.059729,1.130447,6.0,1.060275,0.701082,umass-churn-hpo-260725-2158-009-d35ff02c,Completed,0.98374,2026-07-25 22:06:13+00:00,2026-07-25 22:07:02+00:00,49.0
3,2.581850,0.735833,0.046424,6.062796,6.0,2.062458,0.957600,umass-churn-hpo-260725-2158-007-2808a6e9,Completed,0.98278,2026-07-25 22:05:06+00:00,2026-07-25 22:05:50+00:00,44.0
8,4.920339,0.605700,0.115000,0.551889,7.0,8.446254,0.922388,umass-churn-hpo-260725-2158-002-707d6516,Completed,0.98275,2026-07-25 21:59:23+00:00,2026-07-25 22:01:28+00:00,125.0
2,2.821048,0.949217,0.253524,2.965278,3.0,2.038706,0.738505,umass-churn-hpo-260725-2158-008-96cc80dc,Completed,0.98273,2026-07-25 22:05:10+00:00,2026-07-25 22:05:55+00:00,45.0
4,0.956737,0.687685,0.090594,0.700524,10.0,2.633629,0.719602,umass-churn-hpo-260725-2158-006-a2d7f4b8,Completed,0.98231,2026-07-25 22:04:01+00:00,2026-07-25 22:04:51+00:00,50.0
9,4.624448,0.873097,0.192182,6.913097,10.0,4.454326,0.799538,umass-churn-hpo-260725-2158-001-427d23df,Completed,0.98227,2026-07-25 21:59:22+00:00,2026-07-25 22:01:27+00:00,125.0
7,0.135577,0.773130,0.071226,1.718333,4.0,2.022356,0.722506,umass-churn-hpo-260725-2158-003-ce007b21,Completed,0.98217,2026-07-25 22:02:01+00:00,2026-07-25 22:02:50+00:00,49.0
5,3.686477,0.928704,0.027810,2.510328,9.0,9.456135,0.799816,umass-churn-hpo-260725-2158-005-1060c34b,Completed,0.98210,2026-07-25 22:04:00+00:00,2026-07-25 22:04:49+00:00,49.0
6,2.080651,0.992835,0.026637,5.798032,6.0,9.987450,0.954681,umass-churn-hpo-260725-2158-004-43d987b0,Completed,0.98158,2026-07-25 22:02:02+00:00,2026-07-25 22:02:46+00:00,44.0
0,4.940935,0.655702,0.029660,1.153830,10.0,7.660498,0.978664,umass-churn-hpo-260725-2158-010-58614242,Completed,0.97979,2026-07-25 22:06:15+00:00,2026-07-25 22:07:04+00:00,49.0


In [ ]:
# Hyperparameter tuning identified a moderately complex, strongly regularized XGBoost configuration as the best-performing candidate. 
# The optimal model achieved a validation ROC-AUC of 0.98374 using a maximum tree depth of 6, a low learning rate of approximately 0.06, 
# row subsampling of 70%, and feature subsampling of 79%. These settings balance predictive capability with controls against overfitting. 
# The best trial completed in 49 seconds and outperformed the deeper depth-7 configuration, which required 125 seconds without improving AUC. 
# However, the performance differences among the leading trials were very small, indicating that several configurations lie within a 
# similarly strong performance region. The selected model should therefore be validated on the untouched test set and reviewed for temporal or
# target leakage before it is registered or deployed.